In [110]:
import numpy as np 
from scipy import stats
from scipy.stats import poisson, expon 
import pandas as pd
import math


In [111]:
np.random.seed(2004) # for reducibility 

# 1: POISSON DISTRIBUTION 
The poisson distribution models the number of events occuring in a fixed interval of time or space, given: 
- events are independent 
- average rate (lambda) is constant  
### The PMF Formula
$$P(X = k) = \frac{\lambda^k \cdot e^{-\lambda}}{k!}$$


### Our Scenario
A coffee shop receives on average **5 customer arrivals per hour**. We model this with λ = 5.


In [112]:
lam_poisson = 5  # average arrivals per hour 
poisson_data = np.random.poisson(lam=lam_poisson, size=2000)   # generate 2000 random numbers from a Poisson distribution with mean (Lambda) of 5

print("Parameters:")
print(f"  λ (lambda) = {lam_poisson} arrivals per hour")
print(f"  Sample size = 2000 simulated hours")

print(f"\n First 20 generated numbers: \n {poisson_data[:20]}")  # print the first 20 generated numbers 
# Calculate the sample mean and variance 
print(f"{'Mean ':<10}: {np.mean(poisson_data):.4f}\n{'Variance ':<10}: {np.var(poisson_data, ddof=1):.4f} \ntheoritical mean and variance is: \n{'λ':<10}: {lam_poisson}" )  


Parameters:
  λ (lambda) = 5 arrivals per hour
  Sample size = 2000 simulated hours

 First 20 generated numbers: 
 [9 4 6 8 8 2 6 6 5 3 3 8 7 8 5 1 3 8 3 5]
Mean      : 4.9250
Variance  : 4.7708 
theoritical mean and variance is: 
λ         : 5


To print the very first 20 stimulated values 

## Probability of exactly 3 arrivales 
$$P(X = 3) = \frac{5^3 \cdot e^{-5}}{3!} = \frac{125 \cdot 0.006738}{6}$$

In [113]:
exact_k = 3 
# Manual calculation of P(X = 3) 
numerator = (lam_poisson**exact_k) * math.exp(-lam_poisson) 
denominator =math.factorial(exact_k) 
manual_prob = numerator / denominator 


# Using scipy to calculate P(X = 3) 
scipy_prob =poisson.pmf(k=exact_k, mu=lam_poisson)  

print (f"\nManual calculation of P(X = {exact_k}): {manual_prob:.6f} \nScipy calculation of p(X ={exact_k}): {scipy_prob:.6f}")

if np.isclose(manual_prob, scipy_prob):
    print(f"\nManual and scipy probabilities match: P(X = {exact_k}) = {manual_prob:.6f}") 
else: 
    print(f"\nIssue found: Manual P(X = {exact_k}) = {manual_prob:.6f}, Scipy P(X = {exact_k}) = {scipy_prob:.6f}") 

print (f"\nInterpertation: There is a  {scipy_prob*100:.2f}% Chance  of observing exactly {exact_k} customers to arrive in a given time")





Manual calculation of P(X = 3): 0.140374 
Scipy calculation of p(X =3): 0.140374

Manual and scipy probabilities match: P(X = 3) = 0.140374

Interpertation: There is a  14.04% Chance  of observing exactly 3 customers to arrive in a given time


### Cumulative Probabilities — P(X ≤ k) for k = 0 to 10

The **CDF (Cumulative Distribution Function)** answers: *what is the probability of k OR FEWER arrivals?*

It simply adds up all PMF values from 0 up to k:
$$P(X \leq k) = \sum_{i=0}^{k} P(X = i)$$

In [114]:
# Manual summation of PMF vlaues to give us the wanted value of CDF
cdf = 0
cummulative_k =10
print (f"The CDF for each K value until {cummulative_k}\n")
for i in range (cummulative_k+1):
    pmf_ =(lam_poisson**i) * math.exp(-lam_poisson) / math.factorial(i) 
    cdf+= pmf_
    print(f"{i:<4}: {cdf:.6f}") 
print(f"\n f(X≤10)=  {cdf:.6f}")


# Finding the CDF using scipy.stats 

print(f" using scipy, the CDF, p(x =< {cummulative_k}) = {np.round(poisson.cdf(k=cummulative_k, mu=lam_poisson), 6)}")

The CDF for each K value until 10

0   : 0.006738
1   : 0.040428
2   : 0.124652
3   : 0.265026
4   : 0.440493
5   : 0.615961
6   : 0.762183
7   : 0.866628
8   : 0.931906
9   : 0.968172
10  : 0.986305

 f(X≤10)=  0.986305
 using scipy, the CDF, p(x =< 10) = 0.986305


In [115]:
df = pd.DataFrame({"arrivals": poisson_data})
df.to_csv("poisson_arrivals.csv", index=False)

---
## Section 3: Exponential Distribution

### What Is It?
The Exponential distribution models the **time between successive Poisson events**.

If customers arrive at rate λ = 5 per hour, then the waiting time between arrivals follows an Exponential distribution with:
$$\text{Mean inter-arrival time} = \frac{1}{\lambda} = \frac{1}{5} \text{ hour} = 12 \text{ minutes}$$

### The PDF Formula
$$f(x) = \lambda \cdot e^{-\lambda x}, \quad x \geq 0$$

### The CDF Formula
$$F(x) = P(X \leq x) = 1 - e^{-\lambda x}$$

Therefore, the survival function (probability of waiting MORE than x):
$$P(X > x) = e^{-\lambda x}$$

### Key Property — Memorylessness
$$P(X > s + t \ | \ X > s) = P(X > t)$$

This means: **no matter how long you have already waited, your expected remaining wait time is always the same.**
This is the only continuous distribution with this property.

In [116]:
# ── Parameters ──────────────────────────────────────────────
lambda_exp   = 5              # arrivals per hour (same lambda as Poisson)
mean_minutes = 60 / lambda_exp  # = 12 minutes mean inter-arrival time
rate_per_min = 1 / mean_minutes # λ expressed in per-minute units

# ── Generate 2000 random inter-arrival times (in minutes) ────
# scale = mean = 1/λ (scipy uses scale, not rate)
exp_data_minutes = np.random.exponential(scale=mean_minutes, size=2000)

print("Parameters:")
print(f"  λ = {lambda_exp} arrivals per hour")
print(f"  Mean inter-arrival time = 60 / {lambda_exp} = {mean_minutes} minutes")
print(f"  λ per minute = 1 / {mean_minutes} = {rate_per_min:.4f}")
print()
print("First 20 simulated inter-arrival times (minutes):")
print(np.round(exp_data_minutes[:20], 2))
print()
print("Sample Statistics:")
print(f"  Mean    : {exp_data_minutes.mean():.4f} min  (theoretical: {mean_minutes})")
print(f"  Std Dev : {exp_data_minutes.std():.4f} min  (theoretical: {mean_minutes})")
print()
print("Note: For Exponential, Mean = Std Dev = 1/λ. Another defining property.")

Parameters:
  λ = 5 arrivals per hour
  Mean inter-arrival time = 60 / 5 = 12.0 minutes
  λ per minute = 1 / 12.0 = 0.0833

First 20 simulated inter-arrival times (minutes):
[49.76 15.44  2.43  1.42  6.71  2.65  6.09  0.82  8.22  3.23  1.56  8.07
  3.81  5.25  2.66  3.86 20.29  2.23 20.38  4.4 ]

Sample Statistics:
  Mean    : 12.1844 min  (theoretical: 12.0)
  Std Dev : 11.7852 min  (theoretical: 12.0)

Note: For Exponential, Mean = Std Dev = 1/λ. Another defining property.


### Probability That Next Arrival Takes More Than 15 Minutes

$$P(X > 15) = e^{-\lambda \times 15} = e^{-\frac{1}{12} \times 15} = e^{-1.25}$$

In [117]:
# ── P(X > 15 minutes) ───────────────────────────────────────
t_threshold = 15  # minutes

# Manual step-by-step
exponent     = -rate_per_min * t_threshold
manual_prob  = math.exp(exponent)

# scipy calculation
scipy_prob_more = 1 - expon.cdf(t_threshold, scale=mean_minutes)

print("Step-by-step: P(X > 15 min)")
print(f"  P(X > 15) = e^(-λ × 15)")
print(f"            = e^(-{rate_per_min:.4f} × 15)")
print(f"            = e^({exponent:.4f})")
print(f"            = {manual_prob:.6f}")
print()
print(f"  scipy confirms : P(X > 15) = {scipy_prob_more:.6f}")
print(f"  In percentage  : {scipy_prob_more * 100:.2f}%")
print()
print("Interpretation: There is a 28.65% chance that you wait")
print("more than 15 minutes for the next customer arrival.")

Step-by-step: P(X > 15 min)
  P(X > 15) = e^(-λ × 15)
            = e^(-0.0833 × 15)
            = e^(-1.2500)
            = 0.286505

  scipy confirms : P(X > 15) = 0.286505
  In percentage  : 28.65%

Interpretation: There is a 28.65% chance that you wait
more than 15 minutes for the next customer arrival.


### Memorylessness — Demonstrated Numerically

The memoryless property states that past waiting time is completely irrelevant to future waiting time.

**Scenario:** You have already waited 10 minutes. How much longer do you expect to wait?

**Intuitive (wrong) answer:** "Less than 12 minutes, because I've already waited 10."
**Correct answer:** Still exactly 12 minutes — the distribution resets.

In [ ]:
# ── Memorylessness Demonstration ────────────────────────────
waited = 10  # minutes already waited

print("MEMORYLESSNESS PROPERTY")
print("=" * 50)
print()
print(f"You have already waited {waited} minutes.")
print(f"How much longer do you expect to wait?")
print()
print(f"  Original expected wait time  : {mean_minutes:.2f} minutes")
print(f"  Expected wait AFTER {waited} min  : {mean_minutes:.2f} minutes  ← identical")
print()
print("Mathematical proof:")
print(f"  P(X > {waited} + t | X > {waited}) = P(X > t)")
print(f"  E[X - {waited} | X > {waited}] = 1/λ = {mean_minutes} min")
print()

# Verify numerically — conditional probability
# P(X > 10 + 5 | X > 10) should equal P(X > 5)
t_extra   = 5
p_gt_15   = math.exp(-rate_per_min * (waited + t_extra))
p_gt_10   = math.exp(-rate_per_min * waited)
p_gt_5    = math.exp(-rate_per_min * t_extra)
conditional = p_gt_15 / p_gt_10

print(f"Numerical verification:")
print(f"  P(X > {waited + t_extra} | X > {waited}) = P(X > {waited+t_extra}) / P(X > {waited})")
print(f"                        = {p_gt_15:.6f} / {p_gt_10:.6f}")
print(f"                        = {conditional:.6f}")
print(f"  P(X > {t_extra})              = {p_gt_5:.6f}")
print(f"  Are they equal?       = {abs(conditional - p_gt_5) < 1e-10}")

### Export Exponential Data for Tableau